In [ ]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import math
import random

random.seed(42)
np.random.seed(42)

columns = [
    'Date','Pond_ID','Pond_Type','Target_Species','Season','Water_Temp_C',
    'Weather_Condition','Rainfall_mm','pH_Level','Ammonia_ppm',
    'Dissolved_Oxygen_mgL','Mortality_Count','Fish_Count',
    'Daily_Feed_kg','Est_Avg_Weight_g','Feed_Cost_NPR',
    'Labor_Cost_NPR','Market_Price_NPR','Estimated_Revenue_NPR',
    'Daily_Profit_Loss_NPR','Stocking_Density','Harvest_Ready'
]

start_date = datetime(2022, 1, 1)
num_days = 1460  # 4 years
ponds = [
    ("GrowOut-A", 9800),
    ("GrowOut-B", 10500),
    ("GrowOut-C", 9200),
]

species = "Pangasius"
pond_type = "Farming"

rows = []

def season_from_month(m):
    if m in [12,1,2]:
        return "Winter"
    elif m in [3,4,5]:
        return "Spring"
    elif m in [6,7,8,9]:
        return "Monsoon"
    else:
        return "Autumn"

def weather_from_conditions(temp, rain):
    if rain > 25:
        return random.choice(["Heavy Rain","Stormy","Cloudy"])
    elif rain > 5:
        return random.choice(["Rainy","Cloudy"])
    elif temp > 30:
        return random.choice(["Sunny","Hot"])
    elif temp < 18:
        return random.choice(["Cold","Cloudy","Sunny"])
    return random.choice(["Sunny","Partly Cloudy","Cloudy"])

for pond_id, initial_stock in ponds:
    fish_count = initial_stock
    cycle_day = 0
    cycle_length = random.randint(260, 340)

    for day in range(num_days):
        current_date = start_date + timedelta(days=day)
        doy = current_date.timetuple().tm_yday
        month = current_date.month

        # Restocking after harvest
        if cycle_day >= cycle_length:
            fish_count = random.randint(9000, 12000)
            cycle_day = 0
            cycle_length = random.randint(260, 340)

        season = season_from_month(month)

        # Nepal seasonal temperature curve
        annual_temp = 24 + 8 * math.sin((2 * math.pi * (doy - 110)) / 365)
        temp_noise = np.random.normal(0, 1.2)
        water_temp = round(max(14.5, min(34.5, annual_temp + temp_noise)), 1)

        # Rainfall pattern
        monsoon_strength = max(0, math.sin((2 * math.pi * (doy - 150)) / 365))
        rainfall = max(0, np.random.gamma(2, 6) * monsoon_strength)
        rainfall = round(min(rainfall, 120), 1)

        weather = weather_from_conditions(water_temp, rainfall)

        # pH stable
        ph = round(min(8.8, max(6.5, 7.4 + np.random.normal(0, 0.25))), 1)

        # Logistic fish growth
        max_weight = random.uniform(1100, 1600)
        growth_progress = cycle_day / cycle_length
        logistic_weight = max_weight / (1 + np.exp(-7 * (growth_progress - 0.5)))

        temp_factor = max(0.7, min(1.15, water_temp / 27))
        est_weight = logistic_weight * temp_factor + np.random.normal(0, 8)
        est_weight = round(max(15, est_weight), 1)

        biomass_kg = fish_count * est_weight / 1000

        # Feed relation
        feed_ratio = 0.018 if est_weight < 200 else 0.015 if est_weight < 700 else 0.012
        if water_temp < 20:
            feed_ratio *= 0.75
        daily_feed = biomass_kg * feed_ratio * np.random.uniform(0.92, 1.08)
        daily_feed = round(max(1.0, daily_feed), 1)

        # Ammonia linked to biomass/feed/rainfall
        ammonia = (
            0.015 +
            (daily_feed / 1200) +
            (biomass_kg / 120000) -
            (rainfall / 2500) +
            np.random.normal(0, 0.015)
        )
        ammonia = round(min(1.5, max(0.0, ammonia)), 2)

        # DO inversely related to temperature
        dissolved_oxygen = (
            10.8 -
            (water_temp - 18) * 0.19 -
            ammonia * 1.2 +
            np.random.normal(0, 0.4)
        )

        stress_event = False
        if random.random() < 0.018:
            dissolved_oxygen -= random.uniform(1.5, 3.5)
            ammonia += random.uniform(0.05, 0.25)
            stress_event = True

        dissolved_oxygen = round(min(12, max(2.0, dissolved_oxygen)), 1)
        ammonia = round(min(1.8, max(0.0, ammonia)), 2)

        # Mortality logic
        mortality_rate = 0.00008

        if dissolved_oxygen < 4.5:
            mortality_rate += 0.0025

        if ammonia > 0.4:
            mortality_rate += 0.0018

        if stress_event:
            mortality_rate += 0.001

        mortality = int(max(0, np.random.poisson(fish_count * mortality_rate)))
        mortality = min(mortality, fish_count)

        fish_count -= mortality

        # Costs
        feed_cost = int(round(daily_feed * random.uniform(95, 115)))
        labor_cost = int(round(random.uniform(450, 900)))

        # Market price seasonal drift
        market_price = int(round(
            205 +
            12 * math.sin((2 * math.pi * (doy - 40)) / 365) +
            np.random.normal(0, 5)
        ))
        market_price = max(180, min(260, market_price))

        estimated_revenue = int(round(biomass_kg * market_price))

        operational_cost = feed_cost + labor_cost + random.randint(200, 1200)

        if cycle_day > cycle_length * 0.85:
            profit = int(round(
                estimated_revenue * random.uniform(0.008, 0.02) -
                operational_cost
            ))
            harvest_ready = "Yes"
        else:
            profit = int(round(
                estimated_revenue * random.uniform(0.001, 0.008) -
                operational_cost
            ))
            harvest_ready = "No"

        # Density classification
        density_ratio = biomass_kg / 10000
        if density_ratio < 0.45:
            density = "Low"
        elif density_ratio < 0.95:
            density = "Medium"
        else:
            density = "High"

        rows.append([
            current_date.strftime("%Y-%m-%d"),
            pond_id,
            pond_type,
            species,
            season,
            round(water_temp,1),
            weather,
            rainfall,
            ph,
            ammonia,
            dissolved_oxygen,
            mortality,
            fish_count,
            daily_feed,
            est_weight,
            feed_cost,
            labor_cost,
            market_price,
            estimated_revenue,
            profit,
            density,
            harvest_ready
        ])

        cycle_day += 1

synthetic_df = pd.DataFrame(rows, columns=columns)

# preserve data types
synthetic_df["Mortality_Count"] = synthetic_df["Mortality_Count"].astype(int)
synthetic_df["Fish_Count"] = synthetic_df["Fish_Count"].astype(int)
synthetic_df["Feed_Cost_NPR"] = synthetic_df["Feed_Cost_NPR"].astype(int)
synthetic_df["Labor_Cost_NPR"] = synthetic_df["Labor_Cost_NPR"].astype(int)
synthetic_df["Market_Price_NPR"] = synthetic_df["Market_Price_NPR"].astype(int)
synthetic_df["Estimated_Revenue_NPR"] = synthetic_df["Estimated_Revenue_NPR"].astype(int)
synthetic_df["Daily_Profit_Loss_NPR"] = synthetic_df["Daily_Profit_Loss_NPR"].astype(int)

output_path = ""
synthetic_df.to_csv(output_path, index=False)

print(synthetic_df.shape)
print(synthetic_df.head())
print(f"\nSaved to: {output_path}")
